# **Task 4 : Fine-Tuning BERT on a Kaggle Dataset**

In [ ]:
!pip install transformers datasets --quiet

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

df = pd.concat([train_df, test_df])
df.head()

In [ ]:
df.rename(columns={'text': 'review', 'label': 'sentiment'}, inplace=True)


**1. Data Preprocessing**

In [ ]:
df.dropna(inplace=True)

df = df.sample(10000, random_state=42)

print("Dataset size:", len(df))
print(df['sentiment'].value_counts())

**2. Data Splitting**

In [ ]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.3, random_state=42)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42)

**3. Tokenization**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

**4. Model Building**

In [ ]:
class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

**5. Fine-Tuning**

In [ ]:
def train_model(model, train_loader, optimizer, epochs=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

    return model

**6. Model Evaluation**

In [ ]:
def evaluate_model(model, data_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')

    return accuracy, precision, recall, f1, true_labels, predictions

**EXPERIMENT 1: FULL BERT**

In [ ]:
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
model_full = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
optimizer = AdamW(model_full.parameters(), lr=2e-5)

model_full = train_model(model_full, train_loader, optimizer)

acc1, prec1, rec1, f1_1, y_true1, y_pred1 = evaluate_model(model_full, val_loader)

print("Full BERT:", acc1, prec1, rec1, f1_1)

**EXPERIMENT 2: FREEZE BERT**

In [ ]:
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

model_frozen = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

for param in model_frozen.bert.parameters():
    param.requires_grad = False

optimizer = AdamW(filter(lambda p: p.requires_grad, model_frozen.parameters()), lr=2e-5)

model_frozen = train_model(model_frozen, train_loader, optimizer)

acc2, prec2, rec2, f1_2, y_true2, y_pred2 = evaluate_model(model_frozen, val_loader)

print("Frozen BERT:", acc2, prec2, rec2, f1_2)

**EXPERIMENT 3: LAST 2 LAYERS**

In [ ]:
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

model_partial = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

for name, param in model_partial.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

optimizer = AdamW(filter(lambda p: p.requires_grad, model_partial.parameters()), lr=2e-5)

model_partial = train_model(model_partial, train_loader, optimizer)

acc3, prec3, rec3, f1_3, y_true3, y_pred3 = evaluate_model(model_partial, val_loader)

print("Last 2 Layers:", acc3, prec3, rec3, f1_3)

**Confusion Matrix**

In [ ]:
cm = confusion_matrix(y_true1, y_pred1)

sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Full BERT")
plt.show()

**Comparison Table**

In [ ]:
results = pd.DataFrame({
    "Model": ["Full BERT", "Frozen BERT", "Last 2 Layers"],
    "Accuracy": [acc1, acc2, acc3],
    "Precision": [prec1, prec2, prec3],
    "Recall": [rec1, rec2, rec3],
    "F1 Score": [f1_1, f1_2, f1_3]
})

results

**Analysis**

* Full BERT gives highest accuracy and F1 score.
* Freezing reduces computation but lowers performance.
* Partial fine-tuning balances performance and efficiency.


**Conclusion**


* In this project, BERT was successfully fine-tuned on the IMDB dataset for sentiment analysis.
* Different experiments showed that full fine-tuning provides best performance, while partial tuning reduces computation.
* The model achieved strong classification performance using standard evaluation metrics.
